In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

customers_df = spark.table(
    f"{catalog}.ecommerce_silver.customers"
)

customers_df.printSchema()
customers_df.show(5, truncate=False)

In [0]:
from pyspark.sql.window import Window


#Step 1 — Build the dimension

In [0]:
window_spec = Window.orderBy(customers_df["customer_id"])

gold_dim_customer = (
    customers_df
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_key",
        F.row_number().over(window_spec)
    )
    .select(
        "customer_key",
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
)

#Write Gold

In [0]:
gold_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/gold/dim_customer"

gold_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_gold.dim_customer
USING DELTA
LOCATION '{gold_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_gold.dim_customer"
).show(5, truncate=False)